# PCA on MNIST

KNN on 784 pixel columns, then on a few principal components; pictures of the digits in 2D and 3D; choosing the number of components.

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

## The data

MNIST: 70,000 handwritten digits of 28 x 28 pixels, one column per pixel. The first download takes a minute; after that it is read from a local cache. We keep a random 42,000 images, the size of Kaggle's Digit Recognizer `train.csv`.

In [ ]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
idx = np.random.default_rng(0).choice(len(X), 42000, replace=False)
X, y = X[idx], y[idx].astype(int)
X.shape   # (42000, 784)

In [ ]:
# one row back into a 28 x 28 picture
px.imshow(X[0].reshape(28, 28), color_continuous_scale="gray_r", title=f"label: {y[0]}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.shape, X_test.shape

## KNN on all 784 columns

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

start = time.perf_counter()
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
acc = accuracy_score(y_test, knn.predict(X_test_s))
print(f"784 columns: accuracy {acc:.3f}, {time.perf_counter() - start:.1f} s")

## KNN on 100 principal components

In [ ]:
start = time.perf_counter()
pca = PCA(n_components=100).fit(X_train_s)          # learn the components from the training set only
Z_train, Z_test = pca.transform(X_train_s), pca.transform(X_test_s)
knn = KNeighborsClassifier(n_neighbors=5).fit(Z_train, y_train)
acc = accuracy_score(y_test, knn.predict(Z_test))
print(f"100 components: accuracy {acc:.3f}, {time.perf_counter() - start:.1f} s")

## Accuracy as k grows (a few values; each takes a few seconds)

In [ ]:
rows = []
for k in [1, 2, 3, 5, 10, 20, 50, 100]:
    p = PCA(n_components=k).fit(X_train_s)
    pred = KNeighborsClassifier(n_neighbors=5).fit(p.transform(X_train_s), y_train).predict(p.transform(X_test_s))
    rows.append({"k": k, "accuracy": accuracy_score(y_test, pred)})
pd.DataFrame(rows)

## Pictures: 2D and 3D (drag the 3D plot to turn it; click a digit in the legend to hide it)

In [ ]:
p3 = PCA(n_components=3).fit(X_train_s)
Z = p3.transform(X_test_s)
df = pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1], "PC3": Z[:, 2], "digit": y_test.astype(str)})
order = {"digit": [str(i) for i in range(10)]}
px.scatter(df, x="PC1", y="PC2", color="digit", category_orders=order, opacity=0.5)

In [ ]:
fig = px.scatter_3d(df, x="PC1", y="PC2", z="PC3", color="digit", category_orders=order, opacity=0.6)
fig.update_traces(marker_size=2)
fig

## What PCA learned

In [ ]:
print(p3.explained_variance_)        # the 3 largest eigenvalues
print(p3.explained_variance_ratio_)  # their share of the total variance
print(p3.components_.shape)          # (3, 784): 3 eigenvectors, each with 784 numbers

In [ ]:
# each component is itself a 28 x 28 picture: which pixels it combines
px.imshow(p3.components_[0].reshape(28, 28), color_continuous_scale="RdBu", title="PC1 as an image")

## Choosing k: keep 90% of the variance

In [ ]:
full = PCA().fit(X_train_s)
cum = np.cumsum(full.explained_variance_ratio_)
k90 = int(np.searchsorted(cum, 0.90) + 1)
print("components for 90% of the variance:", k90)
px.line(x=np.arange(1, 785), y=cum, labels={"x": "k", "y": "cumulative explained variance"})

In [ ]:
# scikit-learn can pick k for us: a number between 0 and 1 means "keep this share of the variance"
PCA(n_components=0.90).fit(X_train_s).n_components_